# PSI/TES thermal inertia and albedo `.bin` maps → model-ready rasters

Source: Putzig & Mellon (2007), *Icarus* 191, 68-94 (cite this if you use the maps).
Files: `NBmap2007.bin` / `DBmap2007.bin` (night/day thermal inertia), `NBmsk2007.bin` / `DBmsk2007.bin` (masks),
`AlbMY24map.bin`, `AlbMY25map.bin`, `AlbMY26map.bin` (+ `AlbMY2xmsk.bin` masks).

| Item | Map files | Mask files |
|---|---|---|
| Header | 14400 bytes ASCII | 7200 bytes ASCII |
| Grid | 7200 columns x 3600 rows, 20 px/deg | same |
| Data type | big-endian int16 (`>i2`) | uint8 |
| Thermal inertia | value in tiu (5-5000), 0 = no data | 1 = measured, 0 = infilled |
| Albedo | value / 10000, 0 = no data | 1 = measured, 0 = infilled |

The files were written on a Sun/Solaris machine (big-endian), so on an Intel/ARM computer the bytes must be swapped.
Reading with dtype `">i2"` handles that.

In [11]:
from pathlib import Path
import numpy as np
from scipy.ndimage import map_coordinates

DATA_DIR = Path("../raw_data")            # folder holding the downloaded .bin files
OUT_DIR = Path("../processed_data")        # outputs go here
OUT_DIR.mkdir(exist_ok=True)

NROWS, NCOLS, PPD = 3600, 7200, 20
RES = 1.0 / PPD                    # degrees per pixel (0.05 deg, about 3 km at the equator)
MAP_HEADER, MASK_HEADER = 14400, 7200

# Gale study area (west, south, east, north), lon in degrees East
GALE_BBOX = (136.75, -6.57, 139.24, -4.15)

# ASSUMPTION (verified by the sanity check below): row 0 = 90N, column 0 = 0E, longitude increases east to 360E.
WEST, NORTH = 0.0, 90.0

## 1. Read a map (and its mask)

In [12]:
def read_header(path, nbytes):
    with open(path, "rb") as f:
        return f.read(nbytes).decode("ascii", errors="replace")

def _check_size(path, header, itemsize):
    expected = header + NROWS * NCOLS * itemsize
    actual = Path(path).stat().st_size
    if actual != expected:
        raise ValueError(f"{path}: {actual} bytes, expected {expected}. "
                         "Incomplete download, or an HTML error page saved as .bin?")

def read_map(path, scale=1.0, mask_path=None, drop_infilled=False):
    """Return (float32 array with NaN for no-data, bool array True where measured)."""
    _check_size(path, MAP_HEADER, 2)
    raw = np.memmap(path, dtype=">i2", mode="r", offset=MAP_HEADER, shape=(NROWS, NCOLS))
    arr = np.array(raw, dtype=np.float32)        # copy into native byte order
    arr[arr == 0] = np.nan                       # 0 = null
    arr /= scale

    measured = np.ones(arr.shape, dtype=bool)
    if mask_path is not None:
        _check_size(mask_path, MASK_HEADER, 1)
        m = np.memmap(mask_path, dtype="u1", mode="r", offset=MASK_HEADER, shape=(NROWS, NCOLS))
        measured = np.array(m, dtype=bool)
        if drop_infilled:
            arr[~measured] = np.nan              # same as multiplying by the mask
    return arr, measured

print(read_header(DATA_DIR / "NBmap2007.bin", MAP_HEADER)[:500].strip())  # eyeball the ASCII header

----------------------------------------------------------
THERMAL INERTIA MGS TES 20X20 NIGHTSIDE MAP  2007 Jan 30 
                                                          
Longitudes:         0 W to 360 W (0.05 degree bins)       
Latitudes:        -90 N to  90 N (0.05 degree bins)       
Data Range:    5-5000 tiu (0 = no data)    
Format:          7200 2-byte integers/record (14400 bytes each)
                 3601 records (first is header)           
Source:           TES bolometri


In [13]:
# Thermal inertia (tiu = J m-2 K-1 s-1/2) and albedo (unitless). Keep the infilled pixels for now
# and carry the masks along so you can decide later.
ti_night, ti_night_ok = read_map(DATA_DIR / "NBmap2007.bin", mask_path=DATA_DIR / "NBmsk2007.bin")
ti_day,   ti_day_ok   = read_map(DATA_DIR / "DBmap2007.bin", mask_path=DATA_DIR / "DBmsk2007.bin")

alb, alb_ok = {}, {}
for my in ("MY24", "MY25", "MY26"):
    alb[my], alb_ok[my] = read_map(DATA_DIR / f"Alb{my}map.bin", scale=10000.0,
                                   mask_path=DATA_DIR / f"Alb{my}msk.bin")

layers = {"ti_night": ti_night, "ti_day": ti_day, **{f"albedo_{k.lower()}": v for k, v in alb.items()}}
for name, a in layers.items():
    print(f"{name:14s} min={np.nanmin(a):8.3f}  max={np.nanmax(a):9.3f}  valid={np.isfinite(a).mean():.1%}")

ti_night       min=   4.000  max= 4999.000  valid=96.8%
ti_day         min=   5.000  max= 4999.000  valid=96.8%
albedo_my24    min=   0.067  max=    0.567  valid=96.8%
albedo_my25    min=   0.058  max=    0.675  valid=96.9%
albedo_my26    min=   0.058  max=    0.555  valid=96.8%


In [14]:
# --- Orientation fix. Run ONCE, right after the cell that loads the maps. ---
REF_BRIGHT = [(250, 0), (226, 18), (25, 20), (147, 25)]    # Tharsis, Olympus Mons, Arabia Terra, Elysium
REF_DARK   = [(70, 8), (340, -8), (215, -25), (356, -2)]   # Syrtis Major, Sinus Sabaeus, Mare Cimmerium, Meridiani

_j = np.arange(NCOLS)
COL_MODES = {                       # canonical column j (0-360E) -> column in the file
    "east 0-360":    _j,
    "east -180-180": (_j + NCOLS // 2) % NCOLS,
    "west 0-360":    NCOLS - 1 - _j,
    "west -180-180": (NCOLS - 1 - _j + NCOLS // 2) % NCOLS,
}

def _win(arr, lon, lat, flip, idx, half=30):
    r0 = int((NORTH - lat) / RES)
    c0 = int(((lon % 360) - WEST) / RES)
    rs = np.clip(np.arange(r0 - half, r0 + half + 1), 0, NROWS - 1)
    cs = np.arange(c0 - half, c0 + half + 1) % NCOLS
    if flip:
        rs = NROWS - 1 - rs
    return np.nanmean(arr[np.ix_(rs, idx[cs])])

def _score(flip, idx):
    m = lambda arr, pts: np.mean([_win(arr, lo, la, flip, idx) for lo, la in pts])
    s = (m(alb["MY26"], REF_BRIGHT) - m(alb["MY26"], REF_DARK)) / 0.1 \
        + (m(ti_night, REF_DARK) - m(ti_night, REF_BRIGHT)) / 100.0
    return s if np.isfinite(s) else -np.inf

results = sorted(((_score(f, idx), f, name) for f in (False, True) for name, idx in COL_MODES.items()),
                 key=lambda t: t[0], reverse=True)
for s, f, name in results[:3]:
    print(f"score {s:6.2f}   {'south-up' if f else 'north-up'}, {name}")

best, flip, mode = results[0]
margin = best - results[1][0]
if margin < 0.5:
    print(f"\nNo clear winner (margin {margin:.2f}); nothing changed. Compare against NBmap2007.jpg by eye.")
elif (not flip) and mode == "east 0-360":
    print("\nLayout is already north-up, 0-360E: nothing to change.")
else:
    idx = COL_MODES[mode]
    for a in [ti_night, ti_day, ti_night_ok, ti_day_ok, *alb.values(), *alb_ok.values()]:
        b = a[::-1] if flip else a
        a[:] = b[:, idx]
    print(f"\nRe-oriented all layers from ({'south-up' if flip else 'north-up'}, {mode}) to north-up, 0-360E.")

score   2.27   south-up, west 0-360
score   1.06   north-up, west 0-360
score   0.38   south-up, east 0-360



Re-oriented all layers from (south-up, west 0-360) to north-up, 0-360E.


## 2. Georeference and sanity-check the orientation

In [15]:
def lonlat_to_rowcol(lon, lat):
    """Continuous pixel coordinates (pixel centres sit at integer + 0.5 minus 0.5 -> index space)."""
    col = ((np.asarray(lon, float) % 360.0) - WEST) / RES - 0.5
    row = (NORTH - np.asarray(lat, float)) / RES - 0.5
    return row, col

def sample(arr, lon, lat):
    """Bilinear sample at lon (deg E) / lat (deg N); scalars or arrays."""
    row, col = lonlat_to_rowcol(lon, lat)
    return map_coordinates(arr, [np.atleast_1d(row), np.atleast_1d(col)],
                           order=1, mode="nearest", cval=np.nan)

# Orientation test using well-known regions: Tharsis is dusty (bright, very low TI);
# Syrtis Major is dark (low albedo) with higher TI.
tharsis = (250.0, 0.0)
syrtis  = (70.0, 8.0)
a_alb = alb["MY26"]
t_ti, s_ti = sample(ti_night, *tharsis)[0], sample(ti_night, *syrtis)[0]
t_al, s_al = sample(a_alb, *tharsis)[0], sample(a_alb, *syrtis)[0]
print(f"Tharsis: TI={t_ti:.0f}, albedo={t_al:.3f}   Syrtis: TI={s_ti:.0f}, albedo={s_al:.3f}")
if not (t_ti < s_ti and t_al > s_al):
    print("WARNING: orientation does not look right. Compare against NBmap2007.jpg and adjust "
          "WEST/NORTH (or flip with arr[::-1] / np.roll by 3600 columns for a -180..180 layout).")
else:
    print("Orientation check passed (north-up, 0-360E).")

Tharsis: TI=76, albedo=0.287   Syrtis: TI=171, albedo=0.094
Orientation check passed (north-up, 0-360E).


### Optional preview plot

Kept in its own cell, with matplotlib imported only here, so a broken matplotlib install cannot affect anything else.
It saves a PNG instead of opening a window; open `processed/overview.png` to look at it.
Set `PLOT = False` to skip it.

In [16]:
PLOT = False
if PLOT:
    import matplotlib
    matplotlib.use("Agg")                      # render to a file, no GUI/inline machinery
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
    im0 = ax[0].imshow(np.log10(ti_night[::10, ::10]), extent=[0, 360, -90, 90], cmap="viridis")
    ax[0].set_title("Night TI, log10(tiu)"); fig.colorbar(im0, ax=ax[0])
    im1 = ax[1].imshow(alb["MY26"][::10, ::10], extent=[0, 360, -90, 90], cmap="gray", vmin=0.05, vmax=0.4)
    ax[1].set_title("Albedo MY26"); fig.colorbar(im1, ax=ax[1])
    for a in ax:
        a.set_xlabel("East longitude"); a.set_ylabel("Latitude")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "overview.png", dpi=100)
    print("saved", OUT_DIR / "overview.png")

## 3. Crop to Gale Crater

At 20 px/deg each pixel is about 3 km, so Gale (roughly 154 km across) is only ~50 pixels wide here.
That is the real resolution of this data; resampling to 200 m later just interpolates smoothly and does not add detail.

In [17]:
from rasterio.transform import from_origin

def crop(arr, bbox, pad_deg=0.25):
    w, s, e, n = bbox
    w, s, e, n = w - pad_deg, s - pad_deg, e + pad_deg, n + pad_deg
    c0, c1 = int(np.floor((w - WEST) / RES)), int(np.ceil((e - WEST) / RES))
    r0, r1 = int(np.floor((NORTH - n) / RES)), int(np.ceil((NORTH - s) / RES))
    sub = arr[r0:r1, c0:c1].copy()
    transform = from_origin(WEST + c0 * RES, NORTH - r0 * RES, RES, RES)
    return sub, transform

gale = {}
for name, a in layers.items():
    gale[name], gale_tf = crop(a, GALE_BBOX)

ok_masks = {"ti_night": ti_night_ok, "ti_day": ti_day_ok, **{f"albedo_{k.lower()}": v for k, v in alb_ok.items()}}
print("Gale crop shape:", gale["ti_night"].shape)
for name in layers:
    measured, _ = crop(ok_masks[name].astype(np.float32), GALE_BBOX)
    print(f"{name:14s} mean={np.nanmean(gale[name]):8.3f}  measured (not infilled): {measured.mean():.0%}")

Gale crop shape: (59, 60)
ti_night       mean= 242.088  measured (not infilled): 96%
ti_day         mean= 199.207  measured (not infilled): 99%
albedo_my24    mean=   0.197  measured (not infilled): 39%
albedo_my25    mean=   0.187  measured (not infilled): 43%
albedo_my26    mean=   0.188  measured (not infilled): 51%


## 4. Export as GeoTIFFs

Mars is referenced to a sphere of radius 3,396,190 m (the IAU/Mars 2000 sphere used by most orbital products).
Check that your DEM/slope/aspect `.tif` files use a matching CRS; the reprojection step below handles the rest.

In [18]:
import rasterio
from rasterio.crs import CRS

MARS_CRS = CRS.from_proj4("+proj=longlat +a=3396190 +b=3396190 +no_defs")

def write_tif(path, arr, transform, crs=MARS_CRS):
    with rasterio.open(path, "w", driver="GTiff", height=arr.shape[0], width=arr.shape[1],
                       count=1, dtype="float32", crs=crs, transform=transform,
                       nodata=np.nan, compress="deflate") as dst:
        dst.write(arr.astype("float32"), 1)

for name, a in gale.items():
    write_tif(OUT_DIR / f"gale_{name}_native.tif", a, gale_tf)
print(sorted(p.name for p in OUT_DIR.glob("*.tif")))

['gale_albedo_my24_native.tif', 'gale_albedo_my25_native.tif', 'gale_albedo_my26_native.tif', 'gale_ti_day_native.tif', 'gale_ti_night_native.tif']


## 5. Match your model grid

Point `TEMPLATE` at one of your existing slope/aspect/DEM GeoTIFFs. Each layer is resampled onto exactly that grid,
so everything lines up pixel for pixel and can go into one table.

In [19]:
from rasterio.warp import reproject, Resampling

TEMPLATE = Path("gale_dem.tif")    # <- change to one of your project's .tif files

def to_template(arr, src_transform, template_path, method=Resampling.bilinear):
    with rasterio.open(template_path) as t:
        dst = np.full((t.height, t.width), np.nan, dtype=np.float32)
        reproject(source=arr.astype("float32"), destination=dst,
                  src_transform=src_transform, src_crs=MARS_CRS,
                  dst_transform=t.transform, dst_crs=t.crs,
                  src_nodata=np.nan, dst_nodata=np.nan, resampling=method)
        return dst, t

if TEMPLATE.exists():
    import pandas as pd
    grid = {}
    for name, a in gale.items():
        grid[name], t = to_template(a, gale_tf, TEMPLATE)
        write_tif(OUT_DIR / f"gale_{name}_grid.tif", grid[name], t.transform, t.crs)
    # One tidy table: one row per pixel
    rows, cols = np.indices(grid["ti_night"].shape)
    xs, ys = rasterio.transform.xy(t.transform, rows.ravel(), cols.ravel())
    df = pd.DataFrame({"x": xs, "y": ys, **{k: v.ravel() for k, v in grid.items()}})
    df.to_csv(OUT_DIR / "gale_ti_albedo_grid.csv", index=False)
    print(df.describe().T[["count", "mean", "min", "max"]])
else:
    print(f"{TEMPLATE} not found; skipped. Set TEMPLATE to one of your DEM/slope/aspect .tif files.")

gale_dem.tif not found; skipped. Set TEMPLATE to one of your DEM/slope/aspect .tif files.


## 6. Values at the rover (for the REMS join)

`sample` takes longitude (deg E) and latitude, so you can pass the Curiosity traverse directly.
Because the maps are about 3 km per pixel, the rover will sit in the same pixel for many sols.

In [20]:
# Example: approximate Bradbury Landing (replace with your traverse lon/lat arrays)
lon, lat = np.array([137.4417]), np.array([-4.5895])
for name, a in layers.items():
    print(name, sample(a, lon, lat))

ti_night [359.53284]
ti_day [301.13943]
albedo_my24 [0.24947385]
albedo_my25 [0.24233091]
albedo_my26 [0.22528748]


## Modeling notes

- **Which albedo?** The three maps are from Mars Years 24-26 (1999-2004). Curiosity's REMS data starts in MY31 (2012),
  and dust redistributes albedo between years. Treat albedo as a baseline with real uncertainty. Averaging the three,
  e.g. `np.nanmean(np.stack([gale[f"albedo_my{y}"] for y in (24, 25, 26)]), axis=0)`, is a reasonable start.
  Each map has only 34-50% direct global coverage, so check the "measured" percentages above for Gale.
- **Night vs day thermal inertia.** Night TI is the standard input for thermal models because it is less affected by
  slopes and albedo. Day TI is useful as a second feature, and a large gap between the two suggests subpixel heterogeneity.
- **Apparent thermal inertia.** These values come from a thermal model that assumes a flat surface with uniform properties,
  so the physical values at 200 m can differ from the 3 km averages.
- **Infilled pixels.** Pass `drop_infilled=True` to `read_map` to blank out interpolated pixels (<8% of the surface for TI).
- **Circularity.** These TI/albedo maps come from TES surface temperatures, so validate against THEMIS or REMS rather than TES.